<a href="https://colab.research.google.com/github/ViddaD48/Predicting-Iot-IIoT-attacks-using-Edge-IoT-IIoT-dataset/blob/main/DNN_model_on_Edge_IoT_IIoTdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("ML-EdgeIIoT-dataset.csv", low_memory=False)
df

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6,192.168.0.152,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,MITM
1,6,192.168.0.101,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,MITM
2,6,192.168.0.152,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,MITM
3,6,192.168.0.101,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,MITM
4,6,192.168.0.152,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,MITM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157795,2021 23:24:32.698981000,193.152.82.43,192.168.0.128,0,0,0,0,48729,40690,0,...,0,0,0,0,0,0,0,0,1,DDoS_ICMP
157796,2021 23:24:32.699354000,253.52.1.213,192.168.0.128,0,0,0,0,45657,40702,0,...,0,0,0,0,0,0,0,0,1,DDoS_ICMP
157797,2021 23:24:32.719931000,107.155.221.49,192.168.0.128,0,0,0,0,57686,41423,0,...,0,0,0,0,0,0,0,0,1,DDoS_ICMP
157798,2021 23:24:32.752054000,77.242.58.228,192.168.0.128,0,0,0,0,9555,42379,0,...,0,0,0,0,0,0,0,0,1,DDoS_ICMP


In [4]:
print(df['Attack_type'].unique())
print(df['Attack_type'].value_counts())

['MITM' 'Fingerprinting' 'Ransomware' 'Uploading' 'SQL_injection'
 'DDoS_HTTP' 'DDoS_TCP' 'Password' 'Port_Scanning' 'Vulnerability_scanner'
 'Backdoor' 'XSS' 'Normal' 'DDoS_UDP' 'DDoS_ICMP']
Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64


In [5]:
#encode data labels of attack types

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['Attack_type'] = label_encoder.fit_transform(df['Attack_type'])
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(label_mapping)

{'Backdoor': np.int64(0), 'DDoS_HTTP': np.int64(1), 'DDoS_ICMP': np.int64(2), 'DDoS_TCP': np.int64(3), 'DDoS_UDP': np.int64(4), 'Fingerprinting': np.int64(5), 'MITM': np.int64(6), 'Normal': np.int64(7), 'Password': np.int64(8), 'Port_Scanning': np.int64(9), 'Ransomware': np.int64(10), 'SQL_injection': np.int64(11), 'Uploading': np.int64(12), 'Vulnerability_scanner': np.int64(13), 'XSS': np.int64(14)}


In [6]:
non_numeric_cols = df.select_dtypes(include=['object']).columns
print(non_numeric_cols)

Index(['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4',
       'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query',
       'http.request.method', 'http.referer', 'http.request.full_uri',
       'http.request.version', 'tcp.options', 'tcp.payload', 'tcp.srcport',
       'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.msg', 'mqtt.protoname',
       'mqtt.topic'],
      dtype='object')


In [7]:
df.drop(columns=non_numeric_cols, inplace=True, errors='ignore')
df

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,icmp.unused,http.content_length,http.response,http.tls_port,tcp.ack,...,mqtt.msg_decoded_as,mqtt.msgtype,mqtt.proto_len,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157795,0,0,48729,40690,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2
157796,0,0,45657,40702,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2
157797,0,0,57686,41423,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2
157798,0,0,9555,42379,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2


In [8]:
X = df.drop(['Attack_type', 'Attack_label'], axis=1)
y = df['Attack_type']

In [9]:
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

In [10]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Replace NaN or infinite values with 0 after scaling
X_train_scaled[~np.isfinite(X_train_scaled)] = 0
X_val_scaled[~np.isfinite(X_val_scaled)] = 0
X_test_scaled[~np.isfinite(X_test_scaled)] = 0

In [11]:
from torch.utils.data import TensorDataset

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train.values)

X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val.values)

X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test.values)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [12]:
from torch.utils.data import DataLoader
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [13]:
#Create a Model class
class Model(nn.Module):
    def __init__(self, in_features, h1 = 50, h2=25, out_features=15):
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x


In [14]:
# Determine the number of input features dynamically
num_input_features = X_train_scaled.shape[1]
#Create instance of model
model = Model(in_features=num_input_features)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [16]:
# Check if GPU is available and move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Set random seeds for reproducibility
torch.manual_seed(42) # For CPU
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42) # For all GPUs

num_epochs = 50
best_val_acc = 0

print(f"\n{'='*60}")
print("TRAINING")
print(f"{'='*60}\n")

for epoch in range(num_epochs):
    # ---------- TRAINING PHASE ----------
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        # Move to device
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track metrics
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total

    # ---------- VALIDATION PHASE ----------
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            # Move to device
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'Epoch [{epoch+1}/{num_epochs}] - New best validation accuracy: {val_acc*100:.2f}%')

    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}]')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%')
        print(f'  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc*100:.2f}%')
        print()

print("\n✓ Training complete!")
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")


TRAINING

Epoch [1/50] - New best validation accuracy: 69.54%
Epoch [2/50] - New best validation accuracy: 73.56%
Epoch [3/50] - New best validation accuracy: 74.15%
Epoch [4/50] - New best validation accuracy: 75.35%
Epoch [5/50] - New best validation accuracy: 76.08%
Epoch [5/50]
  Train Loss: 0.6129, Train Acc: 75.98%
  Val Loss:   0.6313, Val Acc:   76.08%

Epoch [6/50] - New best validation accuracy: 76.86%
Epoch [7/50] - New best validation accuracy: 76.99%
Epoch [8/50] - New best validation accuracy: 77.39%
Epoch [9/50] - New best validation accuracy: 77.66%
Epoch [10/50]
  Train Loss: 0.5533, Train Acc: 77.67%
  Val Loss:   0.5927, Val Acc:   77.19%

Epoch [11/50] - New best validation accuracy: 78.48%
Epoch [15/50]
  Train Loss: 0.5290, Train Acc: 78.56%
  Val Loss:   0.5481, Val Acc:   78.25%

Epoch [16/50] - New best validation accuracy: 78.63%
Epoch [17/50] - New best validation accuracy: 78.81%
Epoch [20/50] - New best validation accuracy: 79.00%
Epoch [20/50]
  Train Los

In [18]:
from sklearn.metrics import accuracy_score, classification_report

print(f"\n{'='*60}")
print("TESTING")
print(f"{'='*60}\n")

# Load best model
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Test
all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        # Move inputs to the same device as the model
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        all_predictions.extend(predicted.cpu().numpy()) # Move predictions back to CPU for numpy conversion
        all_labels.extend(labels.cpu().numpy()) # Move labels back to CPU for numpy conversion

# Calculate accuracy
test_acc = accuracy_score(all_labels, all_predictions)

print(f"{'='*60}")
print(f"FINAL TEST RESULTS")
print(f"{'='*60}")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"\n{'='*60}")
print(f"DETAILED CLASSIFICATION REPORT")
print(f"{'='*60}\n")
print(classification_report(all_labels, all_predictions,
                            target_names=label_encoder.classes_,
                            digits=4))

print(f"\n{'='*60}")
print("COMPARISON WITH MACHINE LEARNING")
print(f"{'='*60}")
print(f"Random Forest:  92.00%")
print(f"XGBoost:        92.00%")
print(f"Neural Network: {test_acc*100:.2f}%")
print(f"{'='*60}")


TESTING

FINAL TEST RESULTS
Test Accuracy: 80.16%

DETAILED CLASSIFICATION REPORT

                       precision    recall  f1-score   support

             Backdoor     0.9802    0.9205    0.9494      2039
            DDoS_HTTP     0.5630    0.7514    0.6437      2112
            DDoS_ICMP     0.9996    0.9943    0.9970      2818
             DDoS_TCP     0.9175    0.8951    0.9062      2050
             DDoS_UDP     1.0000    0.9969    0.9984      2900
       Fingerprinting     0.8961    0.6900    0.7797       200
                 MITM     1.0000    0.3745    0.5449       243
               Normal     0.9133    0.9000    0.9066      4860
             Password     0.4784    0.2442    0.3234      1998
        Port_Scanning     0.8628    0.9399    0.8997      2014
           Ransomware     0.7863    0.9515    0.8610      2185
        SQL_injection     0.6846    0.5601    0.6162      2062
            Uploading     0.8561    0.4489    0.5889      2054
Vulnerability_scanner     0.9489 